In [33]:
import pandas as pd
import time
import random
import requests
from bs4 import BeautifulSoup
import re
import json
from tqdm import tqdm
from unidecode import unidecode 
from datetime import datetime

In [34]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/87.0.4280.88 Safari/537.36'}

### Functions

In [35]:
def scrape_pincali(operation="venta"):
    
    # -----------------------------
    # Consultation date
    # -----------------------------
    consultation_date = datetime.now()
    
    # -----------------------------
    # Operation mapping
    # -----------------------------
    operation_type = "sell" if operation == "venta" else "rent"
    
    all_tables = []

    # -----------------------------
    # Loop pages
    # -----------------------------
    for i in tqdm(range(1, 5), desc=f"Scraping Pincali - {operation_type}"):
        
        # -----------------------------
        # URL
        # -----------------------------
        url = f"https://www.pincali.com/inmuebles/propiedades-en-{operation}?page={i}"
        
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, "html.parser")
        
        basic_data = soup.find_all("li", class_="property__component")
        
        # -----------------------------
        # Lists of data
        # -----------------------------
        name, type_, bedrooms, bathrooms, latitude, longitude, street, region, locality, price, surface = [], [], [], [], [], [], [], [], [], [], []
        
        # -----------------------------
        # Loop through each property and extract data
        # -----------------------------
        for elementos in basic_data:
            
            price_tag = elementos.find("div", class_="price")
            price.append(price_tag.get_text(strip=True) if price_tag else None)
            
            name_tag = elementos.find("div", class_="title")
            name.append(name_tag.get_text(strip=True) if name_tag else None)
            
            lat = elementos.get("data-lat")
            lng = elementos.get("data-long")
            
            latitude.append(float(lat) if lat else None)
            longitude.append(float(lng) if lng else None)
            
            location_tag = elementos.find("div", class_="location")
            
            if location_tag:
                location_text = location_tag.get_text(strip=True)
                street.append(location_text)
                
                partes = [p.strip() for p in location_text.split(",")]
                locality.append(partes[-1] if len(partes) > 0 else None)
                region.append(partes[-2] if len(partes) > 1 else None)
            else:
                street.append(None)
                region.append(None)
                locality.append(None)
            
            rec = None
            bath = None
            sup = None
            
            features = elementos.find("div", class_="features")
            
            if features:
                for item in features.find_all("div"):
                    texto = item.get_text(strip=True).lower()
                    
                    numero = re.search(r"\d+", texto)
                    numero = int(numero.group()) if numero else None
                    
                    if "rec" in texto:
                        rec = numero
                    elif "bañ" in texto:
                        bath = numero
                    elif "m²" in texto or "m2" in texto:
                        sup = numero
            
            bedrooms.append(rec)
            bathrooms.append(bath)
            surface.append(sup)
            
            type_.append(None)
        
        # -----------------------------
        # DataFrame for page
        # -----------------------------
        table = pd.DataFrame({
            "name": name,
            "price": price,
            "type": type_,
            "size": surface,
            "bedrooms": bedrooms,
            "bathrooms": bathrooms,
            "latitude": latitude,
            "longitude": longitude,
            "street": street,
            "region": region,
            "locality": locality,
            "consultation_date": consultation_date
        })
        
        all_tables.append(table)

    # -----------------------------
    # Final DataFrame
    # -----------------------------
    table = pd.concat(all_tables, ignore_index=True)

    table["country"] = "Mexico"
    table["source"] = "Pincali"
    table["operation"] = operation_type
    
    return table

In [36]:
def icasas(estado, operation="venta", tipo_inmueble="casas"):
    if operation== "venta":
        base_url = "https://www.icasas.mx/venta/habitacionales-{}-{}-2_3_1_0_0_0/list/p_{}"
    elif operation== "renta":
        base_url = "https://www.icasas.mx/renta/habitacionales-{}-{}-2_3_1_0_0_0/list/p_{}"
    else:
        raise ValueError("Invalid operation type. Use 'venta' or 'renta'.")
    all_data=pd.DataFrame()
    for paginas in tqdm(range(1, 5), desc=f"Scraping iCasas - {operation}, {tipo_inmueble} in {estado}"):
        url= base_url.format(tipo_inmueble,estado, paginas)
        r=requests.get(url, headers=headers)
        soup=BeautifulSoup(r.text, 'html.parser')
        resultados=soup.find_all('li', class_='serp-snippet ad featured')
        oferta, precio, superficie, recamaras, bathrooms, lat, lon = [], [], [], [], [], [], []
        for i in resultados:
            a_tag = i.find('a', class_='detail-redirection')
            oferta.append(a_tag.get_text(strip=True) if a_tag else None)
            superficie.append(i.find('span', class_='areaBuilt').get_text(strip=True) if i.find('span', class_='areaBuilt') else None)
            recamaras.append(i.find('span', class_='rooms').get_text(strip=True) if i.find('span', class_='rooms') else None)
            bathrooms.append(i.find('span', class_='bathrooms').get_text(strip=True) if i.find('span', class_='bathrooms') else None)
            lat_tag = i.find('meta', itemprop='latitude')
            lon_tag = i.find('meta', itemprop='longitude')
            lat.append(lat_tag['content'] if lat_tag else None)
            lon.append(lon_tag['content'] if lon_tag else None)
            precio.append(i.find('div', class_='price').get_text(strip=True) if i.find('div', class_='price') else None)


        
        temp = pd.DataFrame({'name': oferta, 'price':precio, 'size': superficie, 'bedrooms': recamaras, 'bathrooms': bathrooms, 'latitude': lat, 'longitude': lon})
        all_data = pd.concat([all_data, temp], ignore_index=True)
    all_data["consultation_date"] = pd.to_datetime("today")
    all_data["source"] = "icasas"
    all_data["country"] = "Mexico"
    all_data["operation"] = "sell" if operation == "venta" else "rent"
    all_data["type"] = "House" if tipo_inmueble == "casas" else "Apartment"

    return all_data


In [37]:
def scrape_yapo(operation="venta", tipo_immueble="apartamentos"):
    
    data = []
    source_name = "yapo.cl"
    
    # Consultation date
    consultation_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for i in tqdm(range(1, 5), desc=f"Scraping Yapo.cl - {operation} {tipo_immueble}"):
        
        url = f"https://www.yapo.cl/searchresult/bienes-raices-{operation}-{tipo_immueble}?o={i}"
        
        try:
            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code == 200:

                soup = BeautifulSoup(response.text, "html.parser")
                ads = soup.find_all("div", class_="d3-ad-tile__content")

                for ad in ads:

                    details = ad.find_all("li", class_="d3-ad-tile__details-item")

                    bedrooms = None
                    bathrooms = None
                    size = None  

                    # Details: Bathrooms, Bedrooms, Size
                    for li in details:

                        icon = li.find("use")

                        if icon:
                            href = icon.get("xlink:href", "")

                            if "#bed" in href:
                                bedrooms = li.get_text(strip=True)

                            elif "#bath" in href:
                                bathrooms = li.get_text(strip=True)

                            elif "#resize" in href:
                                size = li.get_text(strip=True)

                    item = {
                        "consultation_date": consultation_date,
                        "source": source_name,
                        "type": "Apartment" if tipo_immueble == "apartamentos" else "House",
                        "operation": "sell" if operation == "venta" else "rent",
                        "locality": ad.find("div", class_="d3-ad-tile__location").get_text(strip=True)
                        if ad.find("div", class_="d3-ad-tile__location") else None,
                        "country": "Chile",
                        "name": ad.find("span", class_="d3-ad-tile__title").get_text(strip=True)
                        if ad.find("span", class_="d3-ad-tile__title") else None,
                        "price": ad.find("div", class_="d3-ad-tile__price").get_text(strip=True)
                        if ad.find("div", class_="d3-ad-tile__price") else None,
                        "bedrooms": bedrooms,
                        "bathrooms": bathrooms,
                        "size": size
                    }

                    data.append(item)

            time.sleep(random.uniform(1,3))

        except Exception as e:
            print(f"Error en página {i}: {e}")
            break

    return pd.DataFrame(data)

In [38]:
def scrape_properati(pais: str, operation="alquiler"):
    """
    Scrape Properati listings by country.

    Parameters
    ----------
    pais : str
        'Argentina', 'Ecuador', 'Colombia', 'Peru'

    Returns
    -------
    pd.DataFrame
    """

    # =========================
    # Country → domain mapping
    # =========================
    country_domain_map = {
        "Argentina": "ar",
        "Ecuador": "ec",
        "Colombia": "co",
        "Peru": "pe"
    }

    if pais not in country_domain_map:
        raise ValueError(f"Country must be one of {list(country_domain_map.keys())}")

    domain = country_domain_map[pais]

    # =========================
    # Automatic consultation date
    # =========================
    consultation_date = datetime.today().strftime("%Y-%m-%d")


    all_pages = []

    # =========================
    # LOOP OVER PAGES
    # =========================
    for page in tqdm(range(1, 5), desc=f"Scraping Properati - {pais} - {operation}"):

        url = f"https://www.properati.com.{domain}/s/{operation}/{page}"

        response = requests.get(url, headers=headers)
        time.sleep(random.uniform(1,3))

        if response.status_code != 200:
            print(f"Page {page} failed with status {response.status_code}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")

        # =========================
        # BASIC INFO
        # =========================
        basics = soup.find_all("div", class_="snippet__content")

        if len(basics) == 0:
            continue

        price, surface = [], []

        for element in basics:
            price.append(
                element.find("div", class_="price").text.strip()
                if element.find("div", class_="price") else None
            )
            surface.append(
                element.find("span", class_="properties__area").text.strip()
                if element.find("span", class_="properties__area") else None
            )

        basics_df = pd.DataFrame({
            "price": price,
            "size": surface
        })

        if len(basics_df) > 2:
            basics_df = basics_df.drop([0, 1]).reset_index(drop=True)

        # =========================
        # JSON-LD INFO
        # =========================
        scripts = soup.find_all("script", type="application/ld+json")

        if not scripts:
            continue

        data_json = json.loads(scripts[0].text)[0]["about"]

        name, type_, bedrooms, bathrooms = [], [], [], []
        latitude, longitude, street, region, locality = [], [], [], [], []

        for item in data_json:

            type_.append(item.get("@type"))
            bedrooms.append(item.get("numberOfBedrooms"))
            bathrooms.append(item.get("numberOfBathroomsTotal"))

            geo = item.get("geo", {})
            latitude.append(geo.get("latitude"))
            longitude.append(geo.get("longitude"))

            address = item.get("address", {})
            street.append(address.get("streetAddress"))
            region.append(address.get("addressRegion"))
            locality.append(address.get("addressLocality"))

            name.append(item.get("name"))

        table = pd.DataFrame({
            "name": name,
            "type": type_,
            "street": street,
            "region": region,
            "locality": locality,
            "bedrooms": bedrooms,
            "bathrooms": bathrooms,
            "latitude": latitude,
            "longitude": longitude
        })

        # =========================
        # Merge
        # =========================
        final_df = pd.concat([table, basics_df], axis=1)
        final_df["operation"] = "rent" if operation == "alquiler" else "sell"
        final_df["consultation_date"] = consultation_date
        final_df["country"] = pais
        final_df["source"] = "Properati"

        all_pages.append(final_df)

    if not all_pages:
        return pd.DataFrame()

    return pd.concat(all_pages, ignore_index=True)

In [39]:
def scrape_encuentra24(tipo="venta", tipo_immueble="apartamentos", pais="panama"):
    data = []
    source_name = "encuentra24"
    # Consultation date
    consultation_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for i in tqdm(range(1, 5), desc=f"Scraping Encuentra24 - {pais} - {tipo} - {tipo_immueble}"):
        url = f"https://www.encuentra24.com/{pais}-es/searchresult/bienes-raices-{tipo}-de-propiedades.{i}?q=withcat.bienes-raices-{tipo}-de-propiedades-{tipo_immueble}"
        
        try:
            response = requests.get(url, headers=headers, timeout=10)
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')
                ads = soup.find_all('div', class_='d3-ad-tile__content') 
                
                for ad in ads:
                    details = ad.find_all('li', class_='d3-ad-tile__details-item')
                    
                    bedrooms = None
                    bathrooms = None
                    #Details: Bathrooms and Bedrooms
                    for li in details:
                        icon = li.find('use')
                        if icon:
                            href = icon.get('xlink:href', '')
                            if '#bed' in href:
                                bedrooms = li.get_text(strip=True)
                            elif '#bath' in href:
                                bathrooms = li.get_text(strip=True)
                            elif "#resize" in href:
                                size = li.get_text(strip=True)

                    item = {
                        "consultation_date": consultation_date,
                        "source": source_name,
                        #type if "apartamentos" then "appartment" else "house"
                        "type": "Apartment" if tipo_immueble == "apartamentos" else "House",
                        "operation": "sell" if tipo == "venta" else "rent",
                        "locality": ad.find('div', class_='d3-ad-tile__location').get_text(strip=True) if ad.find('div', class_='d3-ad-tile__location') else None,
                        #País con mayúscula
                        "country": pais.capitalize(),
                        "name": ad.find('span', class_='d3-ad-tile__title').get_text(strip=True) if ad.find('span', class_='d3-ad-tile__title') else None,
                        "price": ad.find('div', class_='d3-ad-tile__price').get_text(strip=True) if ad.find('div', class_='d3-ad-tile__price') else None,
                        "bedrooms": bedrooms,
                        "bathrooms": bathrooms,
                        "size": size
                    }
                    data.append(item)
            
            time.sleep(random.uniform(1, 3))
            
        except Exception as e:
            print(f"Error en página {i}: {e}")
            break

    return pd.DataFrame(data)

In [40]:
def scrape_quinto_andar(city, operation):

    rows = []

    for i in tqdm(range(1, 5), desc=f"Scraping QuintoAndar - {operation} in {city}"):

        url = f"https://www.quintoandar.com.br/{operation}/imovel/{city}?pagina={i}"
        url
        response = requests.get(url, headers=headers)

        soup = BeautifulSoup(response.content, "html.parser")

        scripts = soup.find_all("script", type="application/ld+json")

        data = []

        for s in scripts:
            try:
                j = json.loads(s.string)

                if j.get("@type") in ["House", "Apartment"]:
                    data.append(j)

            except:
                pass

        for item in data:

            url_listing = item.get("url")

            rows.append({
                "name": item.get("name"),
                "street": item.get("address"),
                "size": item.get("floorSize"),
                "bedrooms": item.get("numberOfBedrooms"),
                "bathrooms": item.get("numberOfFullBathrooms"),
                "price": item.get("potentialAction", {}).get("price"),
                "currency": item.get("potentialAction", {}).get("priceCurrency"),
                "type": item.get("@type"),
                #Operation si "alugar" then "rent" else "sell"
                "operation": "rent" if operation == "alugar" else "sell",
                "locality": city,
                "consultation_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "country": "Brazil",
                "source": "QuintoAndar"
            })

    df = pd.DataFrame(rows)

    return df

## Get listings

#### Argentina, Ecuador, Peru, Colombia

In [41]:
countries = ["Argentina", "Ecuador", "Colombia", "Peru"]
operations = ["alquiler", "venta"]
#Loop through countries and operations and concatenate results
properati_df = pd.DataFrame()
for country in tqdm(countries):
    for operation in operations:
        try:
            df = scrape_properati(country, operation)
            properati_df = pd.concat([properati_df, df], ignore_index=True)
        except Exception as e:
            print(f"Error scraping country {country} and operation {operation}: {e}")
properati_df

100%|██████████| 4/4 [01:43<00:00, 25.99s/it]


,name,type,street,region,locality,bedrooms,bathrooms,latitude,longitude,price,size,operation,consultation_date,country,source
0,Casa en Alquiler en Villa Gesell,House,"Paseo 108 502-600, Villa Gesell, B7165, Buenos...",Buenos Aires Costa Atlántica,Villa Gesell,2.0,2.0,-37.258423,-56.974463,$ 150.000,None,rent,2026-03-13,Argentina,Properati
1,Departamento en Alquiler en Villa Gesell,Apartment,"Paseo Ciento Veinticinco 125, Villa Gesell, B7...",Buenos Aires Costa Atlántica,Villa Gesell,2.0,2.0,-37.276382,-56.98152,$ 50.000,45 m²,rent,2026-03-13,Argentina,Properati
2,Departamento en Alquiler en Villa Gesell,Apartment,"Mis Ilusiones, Villa Gesell, B7165, Provincia ...",Buenos Aires Costa Atlántica,Villa Gesell,2.0,2.0,-37.274216,-56.98106,$ 60.000,60 m²,rent,2026-03-13,Argentina,Properati
3,Departamento en Alquiler en Villa Gesell,Apartment,"San Jorge Ii, Villa Gesell, Provincia de Bueno...",Buenos Aires Costa Atlántica,Villa Gesell,1.0,1.0,-37.274216,-56.98106,$ 60.000,45 m²,rent,2026-03-13,Argentina,Properati
4,Departamento en Alquiler en Villa Gesell,Apartment,"Edificio Malena, Villa Gesell, Provincia de Bu...",Buenos Aires Costa Atlántica,Villa Gesell,1.0,1.0,-37.293316,-56.995737,$ 60.000,50 m²,rent,2026-03-13,Argentina,Properati
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,Casa en Venta en La Molina,House,"Monte Azul, La Molina, Perú",Lima,Lima,5.0,3.0,-12.0760962,-76.9284932,"USD580,000",266 m²,sell,2026-03-13,Peru,Properati
956,Casa en Venta en La Molina,House,"MonteAzul Park, La Molina, Perú",Lima,Lima,5.0,3.0,-12.0766034,-76.9273324,"USD600,000",450 m²,sell,2026-03-13,Peru,Properati
957,Apartamento en Venta,Apartment,"Malecón Central, Punta Hermosa, Perú",None,None,4.0,5.0,-12.34130737,-76.82356641,"USD120,000",230 m²,sell,2026-03-13,Peru,Properati
958,Apartamento en Venta en Miraflores,Apartment,"Prolongacion Arenales, Miraflores, Perú",Lima,Lima,4.0,4.0,-12.1068573,-77.0321294,NaN,NaN,sell,2026-03-13,Peru,Properati


#### Mexico

In [42]:
pincali_mx_sell= scrape_pincali(operation="venta")
pincali_mx_rent= scrape_pincali(operation="renta")

Scraping Pincali - rent: 100%|██████████| 4/4 [00:05<00:00,  1.33s/it]


In [43]:
icasas_mx_rent= icasas("distrito-federal","renta")
icasas_mx_sell= icasas("distrito-federal","venta")

Scraping iCasas - renta, casas in distrito-federal: 100%|██████████| 4/4 [00:03<00:00,  1.04it/s]
Scraping iCasas - venta, casas in distrito-federal: 100%|██████████| 4/4 [00:04<00:00,  1.04s/it]


#### Chile

In [44]:
tipo_inmueble=["apartamentos","casas"]
tipo_operacion=["venta", "alquiler"]

yapo_df= pd.DataFrame()
for inmueble in tqdm(tipo_inmueble):
    for operacion in tipo_operacion:
        try:
            df = scrape_yapo(operation=operacion, tipo_immueble="apartamentos")
            yapo_df = pd.concat([yapo_df, df], ignore_index=True)
        except Exception as e:
            print(f"Error scraping Yapo for {inmueble} and operation {operacion}: {e}")
yapo_df


100%|██████████| 2/2 [00:50<00:00, 25.16s/it]


,consultation_date,source,type,operation,locality,country,name,price,bedrooms,bathrooms,size
0,2026-03-13 00:05:24,yapo.cl,Apartment,rent,Ñuñoa,Chile,Departamento Ñuñoa,$550.000,2,2,None
1,2026-03-13 00:05:24,yapo.cl,Apartment,rent,Concepción,Chile,"DEPARTAMENTO Concepción Centro, Concepción",$450.000,1,1,36 m2
2,2026-03-13 00:05:24,yapo.cl,Apartment,rent,Antofagasta,Chile,"Departamento, Edificio Gran Pacífico",$900.000,3,2,None
3,2026-03-13 00:05:24,yapo.cl,Apartment,rent,Santiago,Chile,"Grato departamento con vista 2D, 2B",$470.000,2,2,54 m2
4,2026-03-13 00:05:24,yapo.cl,Apartment,rent,Concepción,Chile,DEPTO CENTRICO 3 DORMITORIOS COLO COLO ENTRE O...,$590.000,3,1,68 m2
...,...,...,...,...,...,...,...,...,...,...,...
235,2026-03-13 00:05:49,yapo.cl,Apartment,rent,Santiago,Chile,ARRIENDA AHORA EN LOS ESPINOS 3356 !!,$420.000,1,1,None
236,2026-03-13 00:05:49,yapo.cl,Apartment,rent,Las Condes,Chile,DEPARTAMENTO Av Las Condes,"UF33,00",2,2,86 m2
237,2026-03-13 00:05:49,yapo.cl,Apartment,rent,Macul,Chile,DEPARTAMENTO Condominio Laguna Centro,$460.000,2,2,48 m2
238,2026-03-13 00:05:49,yapo.cl,Apartment,rent,Santiago,Chile,DEPARTAMENTO Chiloe 1221 Santiago,$290.000,1,1,37 m2


#### Costa Rica, Guatemala, El Salvador, Honduras, Nicaragua, Dom. Republic

In [45]:
countries = ["costa-rica", "guatemala", "el-salvador", "honduras", "nicaragua", "dominicana"]
operations = ["alquiler", "venta"]
encuentra_24_df = pd.DataFrame()
for country in tqdm(countries):
    for operation in operations:
        try:
            df = scrape_encuentra24(tipo=operation, tipo_immueble="apartamentos", pais=country)
            encuentra_24_df = pd.concat([encuentra_24_df, df], ignore_index=True)
        except Exception as e:
            print(f"Error scraping Encuentra24 for {country} and operation {operation}: {e}")
encuentra_24_df


Scraping Encuentra24 - costa-rica - alquiler - apartamentos: 100%|██████████| 4/4 [00:12<00:00,  3.05s/it]
Scraping Encuentra24 - costa-rica - venta - apartamentos: 100%|██████████| 4/4 [00:15<00:00,  3.80s/it]
Scraping Encuentra24 - guatemala - alquiler - apartamentos: 100%|██████████| 4/4 [00:11<00:00,  2.87s/it]
Scraping Encuentra24 - guatemala - venta - apartamentos: 100%|██████████| 4/4 [00:14<00:00,  3.60s/it]
Scraping Encuentra24 - el-salvador - alquiler - apartamentos: 100%|██████████| 4/4 [00:11<00:00,  2.79s/it]
Scraping Encuentra24 - el-salvador - venta - apartamentos: 100%|██████████| 4/4 [00:14<00:00,  3.73s/it]
Scraping Encuentra24 - honduras - alquiler - apartamentos: 100%|██████████| 4/4 [00:13<00:00,  3.37s/it]
Scraping Encuentra24 - honduras - venta - apartamentos: 100%|██████████| 4/4 [00:15<00:00,  3.85s/it]
Scraping Encuentra24 - nicaragua - alquiler - apartamentos: 100%|██████████| 4/4 [00:13<00:00,  3.25s/it]
Scraping Encuentra24 - nicaragua - venta - apartamento

,consultation_date,source,type,operation,locality,country,name,price,bedrooms,bathrooms,size
0,2026-03-13 00:06:16,encuentra24,Apartment,sell,Rohrmoser,Costa-rica,DE OPORTUNIDAD Apartamento de 1 habitación en QBO,"$165,000Oportunidad",1,1,48 m2
1,2026-03-13 00:06:16,encuentra24,Apartment,sell,Sabanilla,Costa-rica,Moderno y Amplio Apartamento Sabanilla,"$159,900",3,2,115 m2
2,2026-03-13 00:06:16,encuentra24,Apartment,sell,Escazú,Costa-rica,Apartamento de Lujo en Venta en Escazú | Vista...,"$478,000",3,2.5,190 m2
3,2026-03-13 00:06:16,encuentra24,Apartment,sell,San Rafael,Costa-rica,Venta de Apartamento (Pent-House) amueblado en...,"$350,000",2,2,152 m2
4,2026-03-13 00:06:16,encuentra24,Apartment,sell,San Pablo,Costa-rica,Se vende Apartamento Condominio Altos de Pal...,"$156,000",2,2,96 m2
...,...,...,...,...,...,...,...,...,...,...,...
713,2026-03-13 00:08:33,encuentra24,Apartment,sell,Distrito Nacional,Dominicana,Apartamento a estrenar en venta en Evaristo Mo...,"$206,540",1,1,70 m2
714,2026-03-13 00:08:33,encuentra24,Apartment,sell,Distrito Nacional,Dominicana,Apartamento de Lujo en Evaristo Morales,"$395,000",3,3.5,161 m2
715,2026-03-13 00:08:33,encuentra24,Apartment,sell,Santo Domingo,Dominicana,La torre residencial más moderna con amenidade...,"$70,000",1,1,50 m2
716,2026-03-13 00:08:33,encuentra24,Apartment,sell,La Altagracia,Dominicana,"Apartamento En Venta Proyecto Vista Cana, Punt...","$149,500",1,1,64 m2


#### Brazil

In [46]:
city= ["sao-paulo-sp-brasil"]
operations = ["alugar", "comprar"]
quinto_andar_df = pd.DataFrame()
for operation in operations:
    try:
        df = scrape_quinto_andar(city[0], operation)
        quinto_andar_df = pd.concat([quinto_andar_df, df], ignore_index=True)
    except Exception as e:
        print(f"Error scraping QuintoAndar for {city[0]} and operation {operation}: {e}")
quinto_andar_df


Scraping QuintoAndar - alugar in sao-paulo-sp-brasil: 100%|██████████| 4/4 [00:03<00:00,  1.02it/s]
Scraping QuintoAndar - comprar in sao-paulo-sp-brasil: 100%|██████████| 4/4 [00:04<00:00,  1.07s/it]


,name,street,size,bedrooms,bathrooms,price,currency,type,operation,locality,consultation_date,country,source
0,"Casa com 4 quartos, 450m² em Butantã, São Paulo","Rua Gaspar Moreira, Butantã, São Paulo",450,4,3,15000,BRL,House,rent,sao-paulo-sp-brasil,2026-03-13 00:08:49,Brazil,QuintoAndar
1,"Apartamento com 2 quartos, 35m² em Jardim Sant...","Avenida Nordestina, Jardim Santo Antonio, São ...",35,2,1,1100,BRL,Apartment,rent,sao-paulo-sp-brasil,2026-03-13 00:08:49,Brazil,QuintoAndar
2,"Casa com 4 quartos, 160m² em Chácara Klabin, S...","Rua João Lopes, Chácara Klabin, São Paulo",160,4,4,14250,BRL,House,rent,sao-paulo-sp-brasil,2026-03-13 00:08:49,Brazil,QuintoAndar
3,"Studio com 1 quarto, 26m² em Vila Madalena, Sã...","Rua Girassol, Vila Madalena, São Paulo",26,1,1,5750,BRL,Apartment,rent,sao-paulo-sp-brasil,2026-03-13 00:08:49,Brazil,QuintoAndar
4,"Studio com 1 quarto, 24m² em Pompeia, São Paulo","Avenida Pompéia, Pompeia, São Paulo",24,1,1,2700,BRL,Apartment,rent,sao-paulo-sp-brasil,2026-03-13 00:08:49,Brazil,QuintoAndar
...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,"Apartamento com 2 dorms, 25m²","Rua Artur Bernardes, Vila Invernada, São Paulo",25,2,1,205000,BRL,Apartment,sell,sao-paulo-sp-brasil,2026-03-13 00:08:56,Brazil,QuintoAndar
92,"Apartamento com 2 dorms, 39m²","Rua Coronel Ferreira Leal, Vila Gomes, São Paulo",39,2,1,399000,BRL,Apartment,sell,sao-paulo-sp-brasil,2026-03-13 00:08:56,Brazil,QuintoAndar
93,"Apartamento com 2 dorms, 140m²","Rua Doutor José Carlos de Toledo Piza, Morumbi...",140,2,2,1500000,BRL,Apartment,sell,sao-paulo-sp-brasil,2026-03-13 00:08:56,Brazil,QuintoAndar
94,"Apartamento com 2 dorms, 82m²","Rua Professor Vahia de Abreu, Vila Olímpia, Sã...",82,2,1,635000,BRL,Apartment,sell,sao-paulo-sp-brasil,2026-03-13 00:08:56,Brazil,QuintoAndar


In [47]:
# Concatenate all dataframes
tabla_final=pd.concat([properati_df, pincali_mx_sell, pincali_mx_rent, icasas_mx_rent, icasas_mx_sell, yapo_df, encuentra_24_df, quinto_andar_df], ignore_index=True)

print("Total de registros:", len(tabla_final))
tabla_final

Total de registros: 2398


,name,type,street,region,locality,bedrooms,bathrooms,latitude,longitude,price,size,operation,consultation_date,country,source,currency
0,Casa en Alquiler en Villa Gesell,House,"Paseo 108 502-600, Villa Gesell, B7165, Buenos...",Buenos Aires Costa Atlántica,Villa Gesell,2.0,2.0,-37.258423,-56.974463,$ 150.000,None,rent,2026-03-13,Argentina,Properati,NaN
1,Departamento en Alquiler en Villa Gesell,Apartment,"Paseo Ciento Veinticinco 125, Villa Gesell, B7...",Buenos Aires Costa Atlántica,Villa Gesell,2.0,2.0,-37.276382,-56.98152,$ 50.000,45 m²,rent,2026-03-13,Argentina,Properati,NaN
2,Departamento en Alquiler en Villa Gesell,Apartment,"Mis Ilusiones, Villa Gesell, B7165, Provincia ...",Buenos Aires Costa Atlántica,Villa Gesell,2.0,2.0,-37.274216,-56.98106,$ 60.000,60 m²,rent,2026-03-13,Argentina,Properati,NaN
3,Departamento en Alquiler en Villa Gesell,Apartment,"San Jorge Ii, Villa Gesell, Provincia de Bueno...",Buenos Aires Costa Atlántica,Villa Gesell,1.0,1.0,-37.274216,-56.98106,$ 60.000,45 m²,rent,2026-03-13,Argentina,Properati,NaN
4,Departamento en Alquiler en Villa Gesell,Apartment,"Edificio Malena, Villa Gesell, Provincia de Bu...",Buenos Aires Costa Atlántica,Villa Gesell,1.0,1.0,-37.293316,-56.995737,$ 60.000,50 m²,rent,2026-03-13,Argentina,Properati,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2393,"Apartamento com 2 dorms, 25m²",Apartment,"Rua Artur Bernardes, Vila Invernada, São Paulo",NaN,sao-paulo-sp-brasil,2,1,NaN,NaN,205000,25,sell,2026-03-13 00:08:56,Brazil,QuintoAndar,BRL
2394,"Apartamento com 2 dorms, 39m²",Apartment,"Rua Coronel Ferreira Leal, Vila Gomes, São Paulo",NaN,sao-paulo-sp-brasil,2,1,NaN,NaN,399000,39,sell,2026-03-13 00:08:56,Brazil,QuintoAndar,BRL
2395,"Apartamento com 2 dorms, 140m²",Apartment,"Rua Doutor José Carlos de Toledo Piza, Morumbi...",NaN,sao-paulo-sp-brasil,2,2,NaN,NaN,1500000,140,sell,2026-03-13 00:08:56,Brazil,QuintoAndar,BRL
2396,"Apartamento com 2 dorms, 82m²",Apartment,"Rua Professor Vahia de Abreu, Vila Olímpia, Sã...",NaN,sao-paulo-sp-brasil,2,1,NaN,NaN,635000,82,sell,2026-03-13 00:08:56,Brazil,QuintoAndar,BRL


In [48]:
### Save to CSV
tabla_final.to_csv("rent_lac_data_example.csv", index=False, encoding="utf-8-sig")